# 02. Array Manipulation & Structural Transformations

This notebook covers array reshaping, dimension swapping, flattening operations (`flatten` vs. `ravel`), multidimensional iteration, joining, and splitting techniques in NumPy.

In [1]:
import numpy as np

> 💡 **Key Concept: Reshaping & Automatic Dimension Inference (`-1`)**
> - **Total Element Invariance:** The total number of elements ($\prod shape$) must remain strictly constant before and after reshaping (e.g., $1\times12 = 3\times4 = 2\times2\times3$).
> - **Shared Memory View:** `.reshape()` returns a **view** whenever possible. Modifying the reshaped array will mutate the original array unless explicitly copied.
> - **Automatic Dimension Inference (`-1`):** Passing `-1` as one of the dimension arguments instructs NumPy to automatically calculate that specific axis length based on the remaining dimensions and total element count.
> - **Constraint & Resilience:** You can use `-1` only **once** per `.reshape()` call. It eliminates hardcoded dimension math, making data pipelines resilient to varying batch sizes or input lengths.

In [2]:
arr=np.array([10,20,30,40,50,60,70,80,90,100,110,120])
new_arr=arr.reshape(4,3) # Reshape the 1D array into a 2D matrix with 4 rows and 3 columns (4x3 = 12 elements)
print("\nReshaped (4x3):\n",new_arr)

# Verify memory ownership (Reference to 01_numpy_basics: .base confirms reshape is a View)
print("Base array (Memory Owner):\n",new_arr.base)

print("\nReshaped (2x6):\n",arr.reshape(2,-1)) # Set rows to 2; '-1' automatically infers the required number of columns (12 / 2 = 6 columns -> 2x6)
print("\nReshaped 3D (3x4x1):\n",arr.reshape(3,4,-1)) # Reshape into a 3D tensor: 3 matrices, 4 rows, and inferred columns (12 / (3*4) = 1 column -> 3x4x1)


Reshaped (4x3):
 [[ 10  20  30]
 [ 40  50  60]
 [ 70  80  90]
 [100 110 120]]
Base array (Memory Owner):
 [ 10  20  30  40  50  60  70  80  90 100 110 120]

Reshaped (2x6):
 [[ 10  20  30  40  50  60]
 [ 70  80  90 100 110 120]]

Reshaped 3D (3x4x1):
 [[[ 10]
  [ 20]
  [ 30]
  [ 40]]

 [[ 50]
  [ 60]
  [ 70]
  [ 80]]

 [[ 90]
  [100]
  [110]
  [120]]]


> 📌 **Takeaway:**
> - As established in `01_numpy_basics`, `.reshape()` generates a **View** whenever possible without allocating new memory.
> - Modifying `new_arr` directly mutates `arr` because `new_arr.base` points to `arr`.

In [4]:
arr_2D=np.array([
                    [7,19,6],
                    [8,20,7]
                ])

# 1. Flattening using .reshape(-1): Collapses 2D array into 1D (returns a View)
matrix1=arr_2D.reshape(-1)
print("1D via reshape(-1):",matrix1)

# Memory ownership check: returns the original array, proving matrix1 is a View
print("Memory owner of matrix1 (View check):",matrix1.base)

# 2. Flattening using .flatten(): Collapses 2D array into 1D (always returns a Copy)
matrix2=arr_2D.flatten()
print("\n1D via .flatten():",matrix2)

# Memory ownership check: returns None, proving matrix2 owns its memory (Copy)
print("Memory owner of matrix2 (Copy check):",matrix2.base)

matrix3=arr_2D.ravel()
# 3. Flattening using .ravel(): Collapses 2D array into 1D (returns a View whenever possible)
print("\nFlattening via .ravel()",matrix3)

# Memory ownership check: returns the original array, proving matrix3 is a View
print("\nMemory owner of matrix3 (View check):",matrix3.base)

1D via reshape(-1): [ 7 19  6  8 20  7]
Memory owner of matrix1 (View check): [[ 7 19  6]
 [ 8 20  7]]

1D via .flatten(): [ 7 19  6  8 20  7]
Memory owner of matrix2 (Copy check): None

Flattening via .ravel() [ 7 19  6  8 20  7]

Memory owner of matrix3 (View check): [[ 7 19  6]
 [ 8 20  7]]


> 📌 **Takeaway:**
> - **`.flatten()`**: Always creates an independent **Copy** in memory (`.base` returns `None`). Safe for non-destructive operations, but costs additional memory allocation.
> - **`.ravel()` & `.reshape(-1)`**: Return a **View** whenever possible (`.base` points to the source array). Highly performance-efficient and memory-friendly for large data pipelines.
> - Always verify memory ownership with `.base` if downstream modifications could inadvertently mutate the original dataset.

### Key Concepts & Memory Logic

- **Axis Parameter (`axis`):**
  - `axis=None`: Reverses elements across **all dimensions** of the array.
  - `axis=0`: Reverses the order of rows vertically (Top <-> Bottom).
  - `axis=1`: Reverses the order of columns horizontally (Left <-> Right).
- **Shorthand Slicing (`[::-1]`):** Standard Python step slicing `arr[::-1]` achieves the exact same reversing behavior as `np.flip(arr)`.
- **Memory Management:** `np.flip()` performs a zero-copy operation by modifying array strides. It returns a **View** sharing the underlying memory buffer with the original array (verified via `.base`).

In [11]:

# 1. 1D Array Reversing
arr_1d = np.array([10, 20, 30, 40, 50])

print("Original 1D Array:", arr_1d)
print("Flipped 1D Array :", np.flip(arr_1d))


# 2. 2D Array Flipping Across Axes
matrix = np.array([
                        [1, 2, 3],
                        [4, 5, 6]
                    ])

print("\nOriginal 2D Matrix:")
print(matrix)

# Vertical Flip (Top <-> Bottom): Reverses rows (axis=0)
flipped_v = np.flip(matrix, axis=0)
print("\nFlipped Vertically (axis=0):")
print(flipped_v)

# Horizontal Flip (Left <-> Right): Reverses columns (axis=1)
flipped_h = np.flip(matrix, axis=1)
print("\nFlipped Horizontally (axis=1):")
print(flipped_h)

# Full Flip: Reverses both axes (axis=None or omitted)
flipped_all = np.flip(matrix)
print("\nFlipped All Axes (axis=None):")
print(flipped_all)


# 3. Memory Ownership Check (View Verification)
print("\nMemory owner of flipped_v (.base check):")
print(flipped_v.base is matrix)  # True -> Confirms it is a View

Original 1D Array: [10 20 30 40 50]
Flipped 1D Array : [50 40 30 20 10]

Original 2D Matrix:
[[1 2 3]
 [4 5 6]]

Flipped Vertically (axis=0):
[[4 5 6]
 [1 2 3]]

Flipped Horizontally (axis=1):
[[3 2 1]
 [6 5 4]]

Flipped All Axes (axis=None):
[[6 5 4]
 [3 2 1]]

Memory owner of flipped_v (.base check):
True


In [13]:

# 1. Standard 2D Matrix (Shape: 2 rows, 3 columns -> (2, 3))
arr_2D = np.array([
    [7, 19, 6],
    [8, 20, 7]
])

print("Original 2D Array (Shape:", arr_2D.shape, "):")
print(arr_2D)

# Transposing via attribute `.T` (Swaps axes: (2, 3) -> (3, 2))
matrix_T = arr_2D.T
print("\nTransposed via .T (Shape:", matrix_T.shape, "):")
print(matrix_T)

# Transposing via method `.transpose()`
matrix_transposed = arr_2D.transpose()
print("\nTransposed via .transpose() (Shape:", matrix_transposed.shape, "):")
print(matrix_transposed)

# Memory ownership check: Transposing produces a View (shares underlying memory)
print("\nMemory owner of matrix_T (View check):")
print(matrix_T.base)


# 2. Advanced: Explicit Axis Permutation in 3D Array (Shape: (2, 2, 3))
arr_3D = np.array([
                    [[1, 2, 3], [4, 5, 6]],
                    [[7, 8, 9], [10, 11, 12]]
                ])

print("\nOriginal 3D Array Shape:", arr_3D.shape) # (depth, rows, cols) -> axes (0, 1, 2)

# Permute axes explicitly: Move axis 2 (cols) to start -> (2, 0, 1)
arr_3D_permuted = arr_3D.transpose(2, 0, 1)
print("Permuted 3D Array Shape (2, 0, 1):", arr_3D_permuted.shape)

Original 2D Array (Shape: (2, 3) ):
[[ 7 19  6]
 [ 8 20  7]]

Transposed via .T (Shape: (3, 2) ):
[[ 7  8]
 [19 20]
 [ 6  7]]

Transposed via .transpose() (Shape: (3, 2) ):
[[ 7  8]
 [19 20]
 [ 6  7]]

Memory owner of matrix_T (View check):
[[ 7 19  6]
 [ 8 20  7]]

Original 3D Array Shape: (2, 2, 3)
Permuted 3D Array Shape (2, 0, 1): (3, 2, 2)


> 📌 **Takeaway:**
> - `np.flip()` mirrors array elements along specified axes without altering the array's shape.
> - Just like `.T` and `.ravel()`, `np.flip()` creates a **View** sharing the same memory buffer, making it O(1) in memory overhead.

### Key Concepts & Memory Logic

- **Joining Mechanism (`np.concatenate`):** Joins a sequence of arrays along an existing axis. All input arrays must have the exact same shape along non-concatenating axes.
- **Convenience Stacking Functions:**
  - `np.stack()`: Joins arrays along a **new axis**, increasing the dimension count by 1 (e.g., two 1D arrays become one 2D array).
  - `np.vstack()` (Vertical Stack): Stacks arrays row-wise along the first axis (`axis=0`).
  - `np.hstack()` (Horizontal Stack): Stacks arrays column-wise along the second axis (`axis=1`). For 1D arrays, concatenates along the first axis.
  - `np.dstack()` (Depth Stack): Stacks arrays along the third dimension (depth / `axis=2`), converting 2D inputs into a 3D array shape `(rows, cols, depth)`.
  - `np.column_stack()`: Stacks 1D arrays as columns into a 2D matrix (equivalent to `np.hstack()` only when inputs are already 2D).
- **Memory Allocation (Deep Copy):** Unlike reshaping or transposing, array joining/stacking operations **allocate a new contiguous memory block**. The resulting array is an independent object, so `.base` returns `None`.

In [32]:
arr1=np.array([19,6,7])
arr2=np.array([1,4,9])

# Concatenating two 1D arrays along axis=0
# For 1D arrays, axis=0 is the only existing dimension, so elements are joined sequentially.
result=np.concatenate((arr1,arr2),axis=0)
print("Concatenated Array (result):",result)

# Two 3-element 1D arrays combine into a single 6-element 1D array.
print("\nShape of Resulting Array (result.shape):",result.shape)
print("\nMemory Owner (.base check):",result.base) # None -> Confirms brand-new memory allocation

arr1_2D=np.array([[91,6,7],
                  [32,4,8]]
                )
arr2_2D=np.array([[35,9,5],
                  [45,72,6]
                 ])

# Vertical Concatenation (axis=0) -> Rows increase: (2, 3) + (2, 3) -> (4, 3)
result1=np.concatenate((arr1_2D,arr2_2D),axis=0)
print("\nConcatenated along axis=0 (Rows):")
print(result1)
print("\nShape of Resulting Array (result1.shape):",result1.shape)
print("\nMemory Owner (.base check):",result1.base) # None -> Confirms brand-new memory allocation

# Horizontal Concatenation (axis=1) -> Columns increase: (2, 3) + (2, 3) -> (2, 6)
result2=np.concatenate((arr1_2D,arr2_2D),axis=1)
print("\nConcatenated along axis=1 (Columns):")
print(result2)
print("\nShape of Resulting Array (result2.shape):",result2.shape)
print("\nMemory Owner (.base check):",result2.base) # None -> Confirms brand-new memory allocation

Concatenated Array (result): [19  6  7  1  4  9]

Shape of Resulting Array (result.shape): (6,)

Memory Owner (.base check): None

Concatenated along axis=0 (Rows):
[[91  6  7]
 [32  4  8]
 [35  9  5]
 [45 72  6]]

Shape of Resulting Array (result1.shape): (4, 3)

Memory Owner (.base check): None

Concatenated along axis=1 (Columns):
[[91  6  7 35  9  5]
 [32  4  8 45 72  6]]

Shape of Resulting Array (result2.shape): (2, 6)

Memory Owner (.base check): None


In [71]:
# 1. 1D Array Stacking Demonstrations
arr1 = np.array([19, 6, 7])  # Shape: (3,)
arr2 = np.array([1, 4, 9])   # Shape: (3,)

# np.stack creates a BRAND-NEW axis (1D + 1D -> 2D)
# Stacking along axis=0: Arrays are stacked as rows -> Shape becomes (2, 3)
res1 = np.stack((arr1, arr2), axis=0)
print("Stacked 1D Arrays along axis=0 (Rows):")
print(res1)
print("\nShape after np.stack (axis=0):")
print(res1.shape)

# Stacking along axis=1: Arrays are stacked as columns -> Shape becomes (3, 2)
res2 = np.stack((arr1, arr2), axis=1)
print("\nStacked 1D Arrays along axis=1 (Columns):")
print(res2)
print("\nShape after np.stack (axis=1):")
print(res2.shape)

# np.hstack on 1D arrays: Concatenates end-to-end along existing axis -> Shape: (6,)
res3 = np.hstack((arr1, arr2))
print("\nHorizontal Stack (hstack) on 1D Arrays:")
print(res3)

# np.column_stack takes 1D arrays and stacks them as columns into a 2D matrix
# Converts two (3,) 1D arrays into a single (3, 2) 2D array
res_col_stack = np.column_stack((arr1, arr2))

# Display the resulting 2D array -> [[19, 1], [6, 4], [7, 9]]
print("Stacked 1D Arrays as Columns (2D Matrix):")
print(res_col_stack)

# Display the transformed array dimensions -> (3, 2)
print("\nShape after column_stack:")
print(res_col_stack.shape)


# 2. 2D Array Stacking Demonstrations
arr1_2D = np.array([[91, 6, 7], 
                    [32, 4, 8]])  # Shape: (2, 3)

arr2_2D = np.array([[35, 9, 5], 
                    [45, 72, 6]])  # Shape: (2, 3)

# np.hstack on 2D: Concatenates horizontally along columns (axis=1) -> Shape: (2, 6)
res6 = np.hstack((arr1_2D, arr2_2D))
print("\nHorizontal Stack (hstack) on 2D Arrays:")
print(res6)
print("\nShape of hstack Result:")
print(res6.shape)

# np.vstack on 2D: Concatenates vertically along rows (axis=0) -> Shape: (4, 3)
res7 = np.vstack((arr1_2D, arr2_2D))
print("\nVertical Stack (vstack) on 2D Arrays:")
print(res7)
print("\nShape of vstack Result:")
print(res7.shape)

# np.dstack on 2D: Stacks along the depth dimension (third axis), returning a 3D array -> Shape: (2, 3, 2)
res8 = np.dstack((arr1_2D, arr2_2D))
print("\nDepth Stack (dstack) on 2D Arrays (Returns 3D):")
print(res8)
print("\nShape of dstack Result (3D Array):")
print(res8.shape)

Stacked 1D Arrays along axis=0 (Rows):
[[19  6  7]
 [ 1  4  9]]

Shape after np.stack (axis=0):
(2, 3)

Stacked 1D Arrays along axis=1 (Columns):
[[19  1]
 [ 6  4]
 [ 7  9]]

Shape after np.stack (axis=1):
(3, 2)

Horizontal Stack (hstack) on 1D Arrays:
[19  6  7  1  4  9]
Column stack 1D Arrays
[[19  1]
 [ 6  4]
 [ 7  9]]

Horizontal Stack (hstack) on 2D Arrays:
[[91  6  7 35  9  5]
 [32  4  8 45 72  6]]

Shape of hstack Result:
(2, 6)

Vertical Stack (vstack) on 2D Arrays:
[[91  6  7]
 [32  4  8]
 [35  9  5]
 [45 72  6]]

Shape of vstack Result:
(4, 3)

Depth Stack (dstack) on 2D Arrays (Returns 3D):
[[[91 35]
  [ 6  9]
  [ 7  5]]

 [[32 45]
  [ 4 72]
  [ 8  6]]]

Shape of dstack Result (3D Array):
(2, 3, 2)


> 📌 **Takeaway:**
> - `np.hstack` and `np.vstack` concatenate along existing dimensions, maintaining array ranks.
> - `np.stack` inserts a new axis, increasing array dimension by 1 (e.g., 1D arrays become 2D).
> - `np.dstack` stacks along the third dimension (depth), converting 2D inputs into a 3D array shape `(rows, cols, depth)`.

>   ⚠️ **NumPy 2.0 Modernization Note:**
> `np.row_stack` has been removed in modern NumPy versions. Always use `np.vstack` for vertical stacking and `np.column_stack` for column-wise 1D stacking.

### Key Concepts & Memory Logic

- **Array Splitting Mechanism (`np.split`):** Divides an array into multiple sub-arrays along a specified axis.
  - **Equal Division (Integer Split):** Passing an integer $N$ splits the array into $N$ equal parts along the specified axis. The array dimension along that axis must be evenly divisible by $N$, otherwise NumPy raises a `ValueError`.
  - **Custom Index Splits (1D/2D Slicing):** Passing a 1D array/list of indices (e.g., `[2, 5]`) splits the array at those specific indices, allowing unequal sub-array shapes.
- **Convenience Splitting Functions:**
  - `np.hsplit()` (Horizontal Split): Splits an array horizontally along columns (`axis=1` for 2D, `axis=0` for 1D).
  - `np.vsplit()` (Vertical Split): Splits an array vertically along rows (`axis=0`). Requires the input array to be at least 2D.
  - `np.dsplit()` (Depth Split): Splits a 3D array along the depth axis (`axis=2`).
- **Memory Management (View Creation):** Unlike `np.concatenate` or `np.stack`, array splitting functions return **Views** of the original array (slices of the underlying memory buffer). Modifying a sub-array mutates the original parent array, and `.base` references the original parent object.

In [85]:
array1 = np.array([18,16,7,32,19,23])  # Shape: (3,)

# 2. Equal Splitting via np.split
# Dividing 6 elements into 3 equal sub-arrays (6 / 3 = 2 elements per sub-array)
res_split = np.split(array1, 3)

# 3. Iterative Output Display
# Iterating through the list of sub-arrays returned by np.split
print("Sub-arrays after equal split (3 parts):")
for i in res_split:
    print(i)

# Alternative indexing approach (equivalent to the loop above):
# print(res_split[0])  # Outputs: [18 16]
# print(res_split[1])  # Outputs: [7 32]
# print(res_split[2])  # Outputs: [19 23]

# 4. Memory Ownership Check (View Verification)
# Demonstrating that sub-arrays are Views referencing the parent array
print("\nMemory Owner Check (.base):")
print(res_split[0].base is array1)  # Outputs 'True', confirming View creation

Sub-arrays after equal split (3 parts):
[18 16]
[ 7 32]
[19 23]

Memory Owner Check (.base):
True


In [126]:

# Creating a 2D matrix with shape (6, 2) -> 6 rows, 2 columns
array2 = np.array([[2, 5],
                   [18, 3],
                   [8, 32],
                   [21, 9],
                   [34, 15],
                   [82, 40]])

# 2. Vertical Splitting along Axis 0 (np.split vs np.vsplit)
# Splitting a (6, 2) array along axis=0 into 3 equal sub-arrays -> Each becomes shape (2, 2)
res_split2 = np.split(array2, 3, axis=0)
print("Row-wise Split via np.split (axis=0, 3 parts):")
for sub_arr in res_split2:
    print(sub_arr)
    print("-" * 15)

# Equivalent shortcut using np.vsplit
res_vsplit = np.vsplit(array2, 3)
print("\nRow-wise Split via np.vsplit (3 parts):")
for sub_arr in res_vsplit:
    print(sub_arr)
    print("-" * 15)


# 3. Horizontal Splitting along Axis 1 (np.split vs np.hsplit)
# Splitting a (6, 2) array along axis=1 into 2 equal sub-arrays -> Each becomes shape (6, 1)
res_split3 = np.split(array2, 2, axis=1)
print("\nColumn-wise Split via np.split (axis=1, 2 parts):")
for sub_arr in res_split3:
    print(sub_arr)
    print("-" * 15)

# Equivalent shortcut using np.hsplit (Fixed iteration target variable to res_hsplit)
res_hsplit = np.hsplit(array2, 2)
print("\nColumn-wise Split via np.hsplit (2 parts):")
for sub_arr in res_hsplit:
    print(sub_arr)
    print("-" * 15)


# 4. Memory Ownership Verification (View Check)
# Verifying that splitting creates Views without copying data
print("Memory Owner Check (.base):")
print(res_hsplit[0].base is array2)  # Outputs 'True', confirming View creation

Row-wise Split via np.split (axis=0, 3 parts):
[[ 2  5]
 [18  3]]
---------------
[[ 8 32]
 [21  9]]
---------------
[[34 15]
 [82 40]]
---------------

Row-wise Split via np.vsplit (3 parts):
[[ 2  5]
 [18  3]]
---------------
[[ 8 32]
 [21  9]]
---------------
[[34 15]
 [82 40]]
---------------

Column-wise Split via np.split (axis=1, 2 parts):
[[ 2]
 [18]
 [ 8]
 [21]
 [34]
 [82]]
---------------
[[ 5]
 [ 3]
 [32]
 [ 9]
 [15]
 [40]]
---------------

Column-wise Split via np.hsplit (2 parts):
[[ 2]
 [18]
 [ 8]
 [21]
 [34]
 [82]]
---------------
[[ 5]
 [ 3]
 [32]
 [ 9]
 [15]
 [40]]
---------------
Memory Owner Check (.base):
True


In [128]:

# Constructing a 3D array with shape (2, 2, 2)
# Axis 0: 2 matrices, Axis 1: 2 rows, Axis 2: 2 columns/depth elements
array3 = np.array([
    [[21, 22], [23, 24]],
    [[31, 32], [33, 34]]
])

print("Original 3D Array Shape:", array3.shape)

# 2. Depth Splitting via np.dsplit
# Splitting along the third axis (axis=2 / depth) into 2 equal sub-arrays
# Equivalent to: np.split(array3, 2, axis=2)
res_dsplit = np.dsplit(array3, 2)

print("\nSub-arrays after np.dsplit (2 parts):")
for idx, sub_arr in enumerate(res_dsplit):
    print(f"\nSub-array {idx + 1} (Shape: {sub_arr.shape}):")
    print(sub_arr)

# 3. Memory Ownership Verification (View Check)
# Verifying that dsplit creates Views rather than Deep Copies
print("\nMemory Owner Check (.base):")
print(res_dsplit[0].base is array3)  # Outputs 'True', confirming View creation

Original 3D Array Shape: (2, 2, 2)

Sub-arrays after np.dsplit (2 parts):

Sub-array 1 (Shape: (2, 2, 1)):
[[[21]
  [23]]

 [[31]
  [33]]]

Sub-array 2 (Shape: (2, 2, 1)):
[[[22]
  [24]]

 [[32]
  [34]]]

Memory Owner Check (.base):
True


> 📌 **Takeaway:**
> - `np.dsplit` splits a 3D array along `axis=2` (depth / third dimension).
> - Splitting a `(2, 2, 2)` array along depth into 2 parts results in two sub-arrays of shape `(2, 2, 1)`.
> - `np.dsplit(arr, N)` is mathematically equivalent to `np.split(arr, N, axis=2)`.
> - Like all splitting operations in NumPy, `np.dsplit` returns **Views** of the original array ($O(1)$ memory cost).

## 📝 Module 02 Summary: Memory Behavior & Golden Rules

### Copy vs. View Behavior in Array Manipulation

| Operation | Functions / Methods | Memory Behavior | `.base` Attribute | Memory Complexity |
| :--- | :--- | :--- | :--- | :--- |
| **Reshaping** | `.reshape()`, `.ravel()` | **View** (usually) | References Parent | $O(1)$ |
| **Flattening** | `.flatten()` | **Deep Copy** | `None` | $O(N)$ |
| **Axis Permutation** | `.T`, `.transpose()`, `.swapaxes()` | **View** | References Parent | $O(1)$ |
| **Reversing** | `np.flip()`, `arr[::-1]` | **View** | References Parent | $O(1)$ |
| **Joining / Stacking** | `np.concatenate()`, `np.vstack()`, `np.hstack()`, `np.stack()`, `np.dstack()`, `np.column_stack()` | **Deep Copy** | `None` | $O(N)$ |
| **Splitting** | `np.split()`, `np.vsplit()`, `np.hsplit()`, `np.dsplit()` | **View** | References Parent | $O(1)$ |

---

> 📌 **Module 02 Golden Rules of Thumb:**
> 1. **Joining Allocates:** Any operation that combines separate memory buffers (`concatenate`, `stack`, `vstack`, etc.) MUST allocate a new contiguous block of memory (**Deep Copy**).
> 2. **Splitting Slices:** Any operation that divides an existing array (`split`, `vsplit`, `hsplit`) merely calculates new offset strides over the existing memory buffer (**View**).
> 3. **Shape Alterations Are Cheap:** Reorienting dimensions (`reshape`, `transpose`, `flip`) modifies metadata (shape/strides) without touching or moving the underlying byte array (**View**).